# Agent Hosting하기

[Notebook 00](./00_The_one_liner_research_agent.ipynb)에서 research agent를 구축했습니다. 노트북 컴퓨터에서 실행됩니다. 이제 팀 동료, cron job, web app 또는 고객 같은 다른 누군가가 사용해야 합니다. 따라서 *자신의 terminal이 아닌 다른 곳*에서 실행되며, 계속 작동하고, 재시작해도 대화를 유지하고, API key를 유출하지 않아야 합니다.

이 Notebook은 정확히 같은 agent를 세 단계로 배포합니다:

| 단계 | 실행 위치 | 사용 시점 |
|---|---|---|
| **1. Docker** | 자신의 컴퓨터 / 단일 VM | 개발 반복 과정, 내부 tool, single-tenant |
| **2. Modal** | Managed serverless | Infra를 관리하지 않고 URL과 scale-to-zero를 원하는 경우 |
| **3. Kubernetes** | 자체 cluster | Multi-tenant, 규제 환경, 완전한 제어 |

Agent 코드, container image, HTTP interface는 세 단계에서 **동일**합니다. Container 주변의 운영 체계만 바뀝니다. 안정적인 interface 뒤에 agent를 container화하면 host 선택은 재작성 대신 배포 결정이 됩니다.

> **이 Notebook 전체 실행 비용:** Anthropic API 호출 약 **$1–2**와 Modal compute **몇 cent**. 모든 단계에 정리 단계가 있습니다.

모든 배포 코드는 이 Notebook 옆의 [`hosting/`](./hosting/)에 있습니다.

## 시작하기 전에: Agent SDK를 사용해야 할까요?

**고객용 chat 제품**을 구축한다면 먼저 [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)를 살펴보세요. Hosting, session, UI가 기본 제공되며 이 Notebook의 대부분을 건너뛸 수 있습니다.

**Programmatic 제어**가 필요한 경우 Agent SDK가 올바른 선택입니다. Batch 및 job 형태의 agent, 내부 tool, 자체 backend에 내장되는 agent, infrastructure를 직접 소유해야 하는 규제 환경 등이 해당합니다. 이런 경우라면 계속 읽으세요.

## 개념 모델

명확히 구분할 세 가지 명사가 있습니다:

- **Process**는 SDK를 load한 실행 중인 Python interpreter 하나입니다. Anthropic API와 통신합니다.
- **Session**은 하나의 대화입니다. 나중에 `resume=`할 수 있도록 SDK가 disk에 기록하는 prompt 기록, tool 호출, 결과입니다.
- **Container**는 package화된 process와 filesystem입니다. Agent 코드, SDK, Node, session 저장 공간을 포함합니다.

Web server 내부에서 "agent" 객체를 instantiate하는 in-process SDK(OpenAI Agents SDK, Google ADK)와 달리, Claude Agent SDK의 agent는 곧 process**입니다**. 따라서 격리는 간단하지만(container 하나 = 영향 범위 하나), hosting은 `pip install` 문제가 아닌 distributed-system 문제입니다.

어느 단계든 모든 배포는 같은 네 가지 작업을 해야 합니다:

```
┌────────────────────────────────────────────────────────────────────┐
│  caller ──► gateway ──► [ spawn | route ] ──► agent container ──► API
│                 │                                     │
│                 └── 인증(agent의 역할 아님)           └── /data  (영속 저장)
│                                                                      │
│  orchestrator ──────────────── lifecycle ────────────────────────────┘
└────────────────────────────────────────────────────────────────────┘
```

1. 작업이 들어오면 container **생성**
2. 각 요청을 해당 session을 소유한 container로 **routing**
3. **Lifecycle**: idle container 종료, crash한 container 재시작
4. 재시작해도 대화를 잃지 않도록 session transcript **영속 저장**

1단계는 네 작업을 모두 수동으로 수행합니다. 2단계는 생성+lifecycle을 Modal에 위임합니다. 3단계는 네 작업 모두를 Kubernetes와 작은 gateway에 위임합니다. Agent container는 바뀌지 않습니다.

## 배포할 Agent

Notebook 00의 [`research_agent/agent.py`](./research_agent/agent.py)를 재사용합니다. `WebSearch`와 `Read`를 가진 한 줄짜리 research agent입니다. Notebook 00을 진행하지 않았다면 먼저 진행하세요. 이 Notebook은 agent가 이미 작동한다고 가정합니다.

`hosting/`이 추가하는 것은 간단한 HTTP server와 Dockerfile뿐입니다. System prompt는 `research_agent.agent`에서 바로 가져오며, 복사하는 대신 import합니다. 한 가지 의도적인 차이가 있습니다. Hosted server는 tool 목록을 `WebSearch`만으로 좁힙니다. Server에 upload 경로가 없으므로 `Read`가 접근할 파일은 다른 session의 transcript와 container 자체 환경뿐이며, prompt injection이 있는 web 결과가 agent를 유도하여 이를 유출하게 할 수 있습니다. `server.py` 주석에 이유가 자세히 설명되어 있습니다.

In [1]:
from research_agent.agent import DEFAULT_MODEL, RESEARCH_SYSTEM_PROMPT

print(f"model: {DEFAULT_MODEL}")
print(RESEARCH_SYSTEM_PROMPT)

model: claude-opus-4-6
You are a research agent specialized in AI.

When providing research findings:
- Always include source URLs as citations
- Format citations as markdown links: [Source Title](URL)
- Group sources in a "Sources:" section at the end of your response


### 설정

API key를 담은 `hosting/.env`를 생성하세요. 이 파일은 gitignore에 포함됩니다.

In [2]:
%%bash
test -f hosting/.env || cp hosting/.env.example hosting/.env
echo 'Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.'
grep -q '^ANTHROPIC_API_KEY=sk-ant-' hosting/.env \
  && ! grep -q 'your-key-here' hosting/.env \
  && echo '✅ key looks set'

Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.


✅ key looks set


---
## 1a단계 — Ephemeral: Prompt 하나, Container 하나, 완료

가능한 한 단순한 배포입니다. 환경 변수의 prompt로 agent를 **한 번** 실행하고 결과를 출력한 뒤 종료하는 container입니다. Server, session, state가 없습니다.


> **Model 참고:** 이 Notebook을 진행하는 동안 test 요청 비용을 낮게 유지하도록 hosting layer의 기본 model은 `claude-sonnet-4-6`입니다. Notebook 00 구성과 정확히 맞추려면 `hosting/.env`에 `MODEL=claude-opus-4-6`을 설정하세요.

이것으로 많은 실제 작업에 충분합니다. Invoice 처리, 야간 보고서 생성, batch 번역, 일회성 분석 등이 해당합니다. Agent의 역할이 "입력 받기, 출력 생성, 중지"라면 이 절 이후 내용은 필요하지 않습니다.

[`Dockerfile`](./hosting/Dockerfile)은 agent, SDK, 그리고 SDK가 내부에서 구동하는 Claude Code CLI를 package화합니다. Image에 `research_agent/`와 `utils/`도 필요하므로 build context는 `hosting/`이 아닌 `claude_agent_sdk/`(현재 디렉터리)입니다:

In [3]:
%%bash
docker build -f hosting/Dockerfile -t research-agent . | tail -n 3

#0 building with "orbstack" instance using docker driver

#1 [internal] load build definition from D

ockerfile
#1 transferring dockerfile: 2.30kB done
#1 DONE 0.0s

#2 resolve image config for docker-i

mage://docker.io/docker/dockerfile:1


#2 DONE 0.6s



#3 docker-image://docker.io/docker/dockerfile:1@sha256:87999aa3d42bdc6bea60565083ee17e86d1f3339802f

...


#12 [5/9] RUN pip install --no-cache-dir -r hosting/requirements.txt
#12 CACHED

#13 [8/9] COPY hos

ting/server.py hosting/run_once.py hosting/entrypoint.sh ./hosting/
#13 CACHED

#14 [9/9] RUN chmod 

+x hosting/entrypoint.sh  && touch hosting/__init__.py
#14 CACHED

#15 exporting to image
#15 export

ing layers done
#15 writing image sha256:b77b20f557cef5e4b9ef01212f3ba3a0895ee97ff8e31f79d9e1dc0cfb7

414f5 done
#15 naming to docker.io/library/research-agent done
#15 DONE 0.0s


In [4]:
%%bash
docker run --rm --env-file hosting/.env \
  -e PROMPT='What is the Claude Agent SDK, in one paragraph?' \
  research-agent

🤖 Thinking...


🤖 Using: ToolSearch()


✓ Tool completed


🤖 Thinking...


🤖 Using: WebSearch()


...

sdk/overview)
- [Building Agents with the Claude Agent SDK – Anthropic Engineering](https://www.an

thropic.com/engineering/building-agents-with-the-claude-agent-sdk)
- [Agent SDK Overview – Anthrop

ic API Docs](https://docs.anthropic.com/en/docs/claude-code/sdk/sdk-overview)
- [claude-agent-sdk-py

thon – GitHub](https://github.com/anthropics/claude-agent-sdk-python)
- [claude-agent-sdk-typescri

pt – GitHub](https://github.com/anthropics/claude-agent-sdk-typescript)


이것으로 끝입니다. `entrypoint.sh`에 `serve` argument가 없으므로 [`run_once.py`](./hosting/run_once.py)가 `$PROMPT`로 `research_agent.agent.send_query()`를 호출하고 0으로 종료합니다.

**이것으로 충분한 경우:** 호출마다 독립적인 job 형태의 작업. Cron, queue worker, CI 단계 또는 CLI를 실행할 수 있는 다른 곳에서 실행하세요.

---
## 1b단계 — Hybrid: 대화를 이어갈 수 있도록 Server 추가하기

Ephemeral mode는 대화를 유지할 수 없습니다. 모든 `docker run`이 새 session으로 시작합니다. Chat 형태의 agent에는 후속 요청을 받고 매번 올바른 session을 resume하는 장기 실행 process가 필요합니다.

[`hosting/server.py`](./hosting/server.py)는 약 100줄의 FastAPI app으로, 정확히 그 일만 합니다. Interface는 모든 단계가 준수하는 contract입니다:

```
GET  /health                            → 200 {"status": "ok"}
POST /sessions/{session_id}/messages    → 200 SDK message의 text/event-stream
```

`server.py`에서 주목할 두 가지 사항:

- **Server에는 인증이 없습니다.** Docstring에 이를 명확히 밝힙니다. 인증은 gateway의 역할입니다(3단계에서 위치를 보여 줌). Server는 `session_id` 형식을 검증하고 호출자를 신뢰합니다.
- **자신의 `session_id`에서 SDK 내부 ID로 연결하는 작은 map을 유지합니다.** SDK는 자체 session ID를 생성하므로 직접 선택할 수 없습니다. Server는 첫 turn의 `ResultMessage`에서 SDK의 ID를 알아내고 후속 요청의 `resume=`에 전달합니다. Map은 `/data` 아래 transcript 옆에 영속적으로 저장됩니다.

Docker-compose로 시작합니다. 재시작해도 transcript가 유지되도록 `./sessions`도 `/data`에 mount합니다:

In [5]:
%%bash
cd hosting/docker && docker compose up --build -d
sleep 3
curl -s http://localhost:8000/health

 Image research-agent Building 


#1 [internal] load local bake definitions


#1 reading from stdin 579B done
#1 DONE 0.0s



#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 2.30kB done


#2 DONE 0.0s

#3 resolve image config for docker-image://docker.io/docker/dockerfile:1


...

 Container docker-research-agent-1 Creating 


 Container docker-research-agent-1 Created 


 Container docker-research-agent-1 Starting 


 Container docker-research-agent-1 Started 


{"status":"ok"}

Prompt를 전송하고 응답을 streaming합니다(`-N`은 curl buffering을 비활성화하여 event가 도착하는 대로 보이게 합니다):

In [6]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"What are the three most interesting AI agent trends right now?"}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01LpCYD1mQgNAmiDYq55RHyC", "name": "WebSearch", "input": {"query": "AI agent trends 2026"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, "usage": {"input_tokens": 685, "cache_cr … [truncated]

[... 6 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "Great question! Based on the latest research and reports, here are the **three most interesting AI agent trends** right now in 2026:\n\n---\n\n## \ud83e\udd1d 1. Multi-Agent Systems & Orchestration\nThe era of the single, all-purpose AI agent is giving way to **coordinat … [truncated]

event: message
data: {"subtype": "s

이제 **같은** `session_id`에 후속 요청을 보냅니다. Server가 session을 resume했으므로 agent가 첫 turn을 기억합니다:

In [7]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Tell me more about the second one."}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01YU5m6b66Kh6GcSN1Kbv7zq", "name": "WebSearch", "input": {"query": "context engineering AI agents 2026 techniques best practices"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, … [truncated]

[... 5 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "## \ud83e\uddf1 Deep Dive: Context Engineering\n\nContext engineering has quickly become **the defining AI skill of 2026**. Here's a thorough breakdown:\n\n---\n\n### What Is It, Exactly?\n\nContext engineering is the discipline of **designing what information an AI mode … [truncated]

event: message
data: {"subtype": "su

Container를 재시작하고 *또 다른* 후속 요청을 보냅니다. Volume mount가 `/data`를 유지했으므로 대화가 살아 있습니다:

In [8]:
%%bash
cd hosting/docker && docker compose restart && sleep 3
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Summarize what we have discussed so far."}'

 Container docker-research-agent-1 Restarting 


 Container docker-research-agent-1 Started 


event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app

", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Gl

ob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "NotebookEdit", "WebFetch", "TodoWrite", "WebS

...

oid context rot and token blowouts.\n- **RAG alone isn't enough** \u2014 77% of IT leaders agree RAG

 is insufficient for production AI on its own.\n- Proper context engineering can improve agent task 

completion rates dramatically (e.g., **83% \u2192 96%**).\n\n---\n\nWould you like to explore any of

 these topics further?", "structured_output": null, "type": "ResultMessage"}



event: done
data: 



In [9]:
%%bash
# 1단계 정리
cd hosting/docker && docker compose down

 Container docker-research-agent-1 Stopping 


 Container docker-research-agent-1 Stopped 
 Container docker-research-agent-1 Removing 


 Container docker-research-agent-1 Removed 


 Network docker_default Removing 


 Network docker_default Removed 


---
## 2단계 — Modal: 같은 Image를 URL로 제공하기

1단계는 자신의 컴퓨터에서 실행됩니다. 2단계는 **같은 Dockerfile**을 `modal.Sandbox`를 통해 [Modal](https://modal.com)에서 실행하며, public HTTPS URL과 scale-to-zero를 제공하고 관리할 server가 없습니다.

그 URL은 *public*입니다. URL을 가진 누구나 API 예산을 사용할 수 있습니다. 1/3단계는 앞에 인증 gateway가 있다고 가정합니다. 2단계에는 gateway가 없으므로 `modal_app.py`가 배포별 bearer token을 생성해 `AGENT_AUTH_TOKEN`으로 전달합니다. `server.py`는 해당 환경 변수가 설정되었을 때만 token을 검사하므로 다른 단계에는 영향이 없습니다.

Agent는 바뀌지 않으므로 [`hosting/modal/modal_app.py`](./hosting/modal/modal_app.py)는 짧습니다:

```python
app = modal.App.lookup("research-agent-hosting", create_if_missing=True)
image = modal.Image.from_dockerfile("hosting/Dockerfile", context_dir=".")

auth_token = secrets.token_urlsafe(32)
sandbox = modal.Sandbox.create(
    "serve",  # compose의 `command:`처럼 image의 ENTRYPOINT에 추가됨
    app=app,
    image=image,
    secrets=[
        modal.Secret.from_name("anthropic"),
        modal.Secret.from_dict({"AGENT_AUTH_TOKEN": auth_token}),
    ],
    volumes={"/data": sessions_volume},
    encrypted_ports=[8000],
)
print(sandbox.tunnels()[8000].url)
```

영속성에는 `/data`에 mount한 `modal.Volume`을 사용하며, 1단계와 같은 `CLAUDE_CONFIG_DIR` 활용 방식입니다. (Workload에서 많은 sandbox가 동시에 쓰고 Volume commit-semantics 문제를 만나면 [`SessionStore`](https://code.claude.com/docs/en/agent-sdk/session-storage)로 교체하세요. 3단계와 production 배포도 이를 사용합니다.)

**Terminal에서** 한 번 설정합니다(`modal setup`은 browser를 열므로 Notebook cell에서 실행할 수 없습니다):

```bash
pip install modal
modal setup
```

그런 다음 Modal이 `ANTHROPIC_API_KEY`로 주입할 secret을 생성합니다:

In [10]:
%%bash
modal secret create anthropic ANTHROPIC_API_KEY="$(grep ANTHROPIC_API_KEY hosting/.env | cut -d= -f2)"

Created a new secret 'anthropic' with the key 'ANTHROPIC_API_KEY'



Use it in your Modal app:




@app.[38;2;248;248;242;48;

2;39;40;34mfunction(secret

s=[[38;2;248;248;242;4

...

                 
    os[

0m.getenv[38;2;248;248;242

;48;2;39;40;34m("ANTHROPIC

_API_KEY")[48;2;39;40

;34m                                              
                             

In [11]:
%%bash
python hosting/modal/modal_app.py | tee /tmp/modal_deploy.out
MODAL_URL=$(awk '/^url:/ {print $2}' /tmp/modal_deploy.out)
MODAL_TOKEN=$(awk '/^token:/ {print $2}' /tmp/modal_deploy.out)
{ echo "MODAL_URL=$MODAL_URL"; echo "MODAL_TOKEN=$MODAL_TOKEN"; } > /tmp/modal_url.env

sandbox: sb-7R7zQ7TtX0h9eKZ8qslvwo
url:     https://ta-01ks91e217n9fymaxjtdc9k5bh-8000-kn9n102ljd7y4

majwav00kwg0.w.modal.host
token:   sb-…redacted…

⚠️  The URL is p

ublic. The token is the only thing gating it — don't share both.

Try it:
  curl -N -X POST https:

//ta-01ks91e217n9fymaxjtdc9k5bh-8000-kn9n102ljd7y4majwav00kwg0.w.modal.host/sessions/demo-1/messages

 \
    -H 'Authorization: Bearer sb-…redacted…' \
    -H 'Content-Type

: application/json' \
    -d '{"prompt":"What are the latest AI agent trends?"}'


In [12]:
%%bash
source /tmp/modal_url.env
curl -N -s -X POST "$MODAL_URL/sessions/demo-1/messages" \
  -H "Authorization: Bearer $MODAL_TOKEN" \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Give me a one-sentence summary of the Claude Agent SDK."}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "1566ffe4-2b20-4a68-82ff-283984b64451", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01PPS8yzMBzMnRhG2Hnpk5VL", "name": "WebSearch", "input": {"query": "Claude Agent SDK Anthropic 2026"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, "usage": {"input_tokens": 120 … [truncated]

[... 5 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "The **Claude Agent SDK** is Anthropic's framework that gives developers programmatic access to the same tools, agent loop, and context management that power Claude Code \u2014 enabling the creation of AI agents that can autonomously read files, run commands, search the w … [truncated]

event: message
data: {"subtype": "s

같은 interface와 image, 다른 host입니다. 호출이 없으면 Modal이 sandbox를 zero로 scale하며 비용이 들지 않습니다.

Idle resource에 과금되지 않도록 정리하세요:

In [13]:
%%bash
python hosting/modal/teardown.py

terminating sandbox sb-7R7zQ7TtX0h9eKZ8qslvwo
deleted volume research-agent-sessions


---
## 3단계 — Kubernetes: Infrastructure를 직접 소유해야 할 때

3단계는 multi-tenant production, 규제 환경 또는 networking, 격리, 비용에 대한 완전한 제어가 필요한 곳을 위한 것입니다. Agent image와 interface는 여전히 동일하며, 새로 추가되는 것은 그 *주변*의 운영 체계입니다:

- 호출자를 인증하고 소유한 `session_id`만 전달하는 앞단 **gateway**. 여기서 `server.py`가 생략한 인증이 이루어집니다.
- 각 대화에 자체 영향 범위를 제공하는 **session별 pod** routing(gateway → Redis → pod).
- 새로운 session이 cold-start latency를 겪지 않도록 미리 준비한 pod의 **standby pool**.
- Prompt injection을 당한 agent가 `api.anthropic.com`에만 접근하게 하는 **egress 제한**(NetworkPolicy + allowlisting proxy).

전체 manifest, gateway 및 단계별 architecture 설명은 [`hosting/kubernetes/`](./hosting/kubernetes/)에 있습니다. Cloud 계정 없이 로컬 [kind](https://kind.sigs.k8s.io/) cluster에서 전체 과정을 실행합니다:

```bash
cd hosting/kubernetes
ANTHROPIC_API_KEY=sk-ant-... ./kind-quickstart.sh
```

Quickstart는 두 demo tenant(`alice`, `bob`)의 bearer token을 출력합니다. Gateway는 각 session의 범위를 생성 tenant로 제한합니다. 1/2단계와 같은 `curl -N POST /sessions/{id}/messages`를 이제 `:8080`의 gateway로 보내며 `Authorization: Bearer <alice-token>` header를 포함하면 alice에게는 작동하지만, 같은 요청에 bob의 token을 사용하면 403을 받습니다. README의 *자체 Cluster에 배포하기* 절에서 실제 cluster를 위한 registry/ingress 변경을 다룹니다.

*Kubernetes 단계는 Joe Shamon과 Ben Lehrburger의 내부 작업을 기반으로 합니다.*


---
## Production에 사용할 수 있도록 만들기

각각 몇 줄로 연결할 수 있는 두 가지 production 고려사항입니다. 아래 cell은 코드를 보여 주며, [hosting 문서](https://code.claude.com/docs/en/agent-sdk/hosting)는 전체 production checklist(인증, graceful shutdown, idle-timeout 조정, autoscaling, 비용 제어)를 다룹니다.

### Observability

SDK는 모든 turn 및 tool 호출에 OpenTelemetry span을 내보냅니다. `server.py`를 변경하지 않고 환경 변수 두 개로 collector를 지정합니다([문서](https://code.claude.com/docs/en/agent-sdk/observability)):

In [14]:
# docker-compose.yml / modal_app.py / k8s Deployment에서:
#   OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
#   OTEL_SERVICE_NAME=research-agent

### Liveness

`server.py`에는 이미 `GET /health`가 있습니다. Orchestrator의 liveness probe를 여기에 지정하세요(compose `healthcheck:`, Modal health check, k8s `livenessProbe`).

### Volume을 넘어서는 영속성

`/data` volume mount는 단일 host 및 Modal에 적합합니다. Multi-host production에서는 transcript를 공유 storage(S3, Postgres, Redis)에 mirror하는 [`SessionStore` adapter](https://code.claude.com/docs/en/agent-sdk/session-storage)를 사용하세요. SessionStore는 mirror라는 점에 주의하세요. 로컬 disk 쓰기가 항상 먼저 발생하며, mirror 실패는 agent를 중단하지 않고 `mirror_error`를 내보냅니다.

### Wire format

`server.py`는 raw SDK message type을 streaming합니다. Cookbook에는 괜찮지만, 실제 API에는 SDK version 변경이 client를 깨뜨리지 않도록 안정적인 wire format을 정의할 것입니다.

---
## 단계 선택하기

| 단계 | 제공하는 것 | 선택할 때 | 다음 단계로 넘어갈 때 |
|---|---|---|---|
| **1. Docker** | 직접 제어하는 컴퓨터의 container. Compose가 재시작하고 bind mount가 `/data`를 유지합니다. Docker host 하나를 운영합니다. | 개발 반복 과정, 내부 tool, single-tenant app, cron/batch job. 수동으로 재시작할 수 있으며 network 외부에서는 호출하지 않습니다. | 컴퓨터 외부의 누군가에게 URL이 필요하거나 "수동으로 재시작"이 더 이상 허용되지 않을 때. |
| **2. Modal** | Public HTTPS URL 뒤의 같은 image, scale-to-zero, 원격 build. 직접 운영할 것은 없습니다. | 즉시 호출 가능한 endpoint가 필요하고, traffic이 순간적으로 몰리거나 대부분 없으며, 배포별 bearer token 인증으로 충분할 때. | 실제 multi-tenant 격리, network 수준 egress 제어가 필요하거나 platform팀이 자체 cluster의 workload를 요구할 때. |
| **3. Kubernetes** | Session별 pod 격리, tenant별 session 범위의 인증 gateway, egress 제한, warm standby pool. Cluster, gateway, Redis를 운영합니다. | Multi-tenant production, 규제 환경 또는 이미 Kubernetes를 운영하며 agent도 다른 요소와 같은 운영 model을 따르게 하려는 경우. | 이 Notebook의 최고 단계입니다. 여기서는 migration 대신 autoscaling, multi-region routing, durable session store를 조정합니다. |

[Hosting guide](https://code.claude.com/docs/en/agent-sdk/hosting)의 배포 pattern은 이 단계들에 직접 대응합니다. **Ephemeral session**(pattern 1: prompt 하나, process 하나, 종료)은 1a단계입니다. Server 없이 어떤 host에서든 cron이나 queue worker로 container를 실행합니다. **Long-running** 및 **hybrid session**(pattern 2/3: process가 turn 간 대화 state를 유지하거나 복원)은 `server.py`가 `resume=`와 `/data` volume으로 구현하는 방식입니다. 1b/2/3단계 모두 이 형태를 제공하며 process를 유지하는 주체와 호출자의 접근 방식만 다릅니다. **단일 Container**(pattern 4: 하나의 container에 여러 session multiplexing)는 정확히 1b/2단계가 하는 일입니다. 이것이 더 이상 허용되지 않고 session별로 자체 영향 범위가 필요할 때를 위해 3단계가 있습니다.


---
## 부록 — 다른 Provider로 Port하기

같은 `hosting/Dockerfile`에 다른 배포 command를 사용합니다. 각각 port 8000을 노출하고 URL을 제공합니다. 영속성을 위해 `/data`에 저장 공간을 mount하세요.

**Fly Machines**
```bash
fly launch --dockerfile hosting/Dockerfile --no-deploy  # claude_agent_sdk/에서 실행
fly volumes create data --size 1
fly deploy
```

**E2B**
```python
from e2b import Sandbox
sbx = Sandbox(template="research-agent")  # hosting/Dockerfile로 build한 template
sbx.commands.run("./hosting/entrypoint.sh serve", background=True)
url = sbx.get_host(8000)
```

**Daytona**
```python
from daytona import Daytona, CreateSandboxFromImageParams
sbx = Daytona().create(CreateSandboxFromImageParams(image="research-agent"))
sbx.process.exec("./hosting/entrypoint.sh serve")
```

**Cloudflare Containers**
```ts
// wrangler.toml은 hosting/Dockerfile을 지정합니다
export class Agent extends Container { defaultPort = 8000 }
```

**Vercel Sandbox**
```ts
import { Sandbox } from "@vercel/sandbox";
const sbx = await Sandbox.create({ image: "research-agent", ports: [8000] });
await sbx.runCommand({ cmd: "./hosting/entrypoint.sh", args: ["serve"], detached: true });
```
